# 01 · Setup do Catalog, Schemas e Volume de Staging

Cria toda a estrutura do Unity Catalog usada pelo projeto **AntecipeAI**:

- 1 catalog dedicado (`antecipeai`, configurável no `.env`)
- 4 schemas **irmãos** dentro dele: `landing`, `bronze`, `silver`, `gold`
- 1 Volume (`raw`) dentro do schema `landing`, usado como staging area
para os arquivos que chegam (hoje: extração única em `.xlsx` da
Locaweb; futuramente: extrações incrementais)

**Decisão de arquitetura:** o schema `landing` guarda o Volume de arquivos
crus (staging), separado da camada `bronze` (que já é Delta). Isso não
estava 100% explícito na conversa anterior — se preferirem outro nome ou
organização para esse schema de staging, é só ajustar `SCHEMA_LANDING` no
`.env` e rodar este notebook de novo (é idempotente, usa `IF NOT EXISTS`
em tudo).

Este notebook é seguro para rodar mais de uma vez (idempotente).

In [0]:
%run ./00_config

## Criar o catalog

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"USE CATALOG {CATALOG}")
print(f"Catalog '{CATALOG}' pronto e selecionado como catalog ativo.")

## Criar os schemas (camadas) — landing, bronze, silver, gold

O DDL exato muda conforme `ANTECIPEAI_TABLE_TYPE` no `.env`:
- `MANAGED` (default do MVP): sem `LOCATION`, o metastore decide onde gravar.
- `EXTERNAL`: adiciona `MANAGED LOCATION` apontando para o bucket/container
configurado em `ANTECIPEAI_STORAGE_ROOT` — é a troca que vamos fazer se o
projeto for aprovado e migrar para S3.

In [0]:
for schema in [SCHEMA_LANDING, SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD]:
    ddl = create_schema_sql(schema)
    print(f"Executando: {ddl}")
    spark.sql(ddl)

print("\nSchemas criados/confirmados:", [SCHEMA_LANDING, SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD])

## Criar o Volume de staging (`landing.raw`)

É para dentro deste Volume que o arquivo `LW-DATASET.xlsx` (ou, no futuro,
as extrações incrementais em CSV) deve ser enviado manualmente via
**"Upload to this volume"** na UI do Catalog Explorer, antes de rodar o
notebook `02_bootstrap_landing_convert_xlsx`.

In [0]:
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_LANDING}.{VOLUME_RAW}
""")

print(f"Volume pronto em: {volume_path()}")
print("Envie o arquivo LW-DATASET.xlsx para esse volume (Catalog Explorer > Upload) antes do próximo notebook.")

## Conferência final da estrutura

In [0]:
print("=== Schemas em", CATALOG, "===")
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

print("=== Volumes em", f"{CATALOG}.{SCHEMA_LANDING}", "===")
display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.{SCHEMA_LANDING}"))